# Fine-tune Llama for translation

We adapt Llama 3.2 (3B) to translate from English to Dutch using sentence pairs from Agatha Christie’s *The ABC Murders*. We use LoRA, which trains a small set of additional weights called an **adapter** while keeping the original model weights fixed.

Open this notebook in Google Colab and select a **T4 GPU** under **Runtime → Change runtime type**. Put the supplied `christie-abc_murders.txt` in the top level of **My Drive**, then choose **Run all**. Allow Colab to access Drive when prompted. The first run downloads the base model.

In [1]:
%pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.12.0 bitsandbytes==0.49.2 datasets==4.4.2 trl==0.25.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 1. Load the translation pairs

Each line contains an English sentence and its Dutch translation, separated by ` ||| `. We give the trainer the English prompt and Dutch answer separately, so it can learn from the answer.

In [2]:
from pathlib import Path

from datasets import Dataset
from google.colab import drive

drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive")
data_path = drive_dir / "christie-abc_murders.txt"
adapter_dir = drive_dir / "lora_model"

if not data_path.is_file():
    raise FileNotFoundError(f"Upload christie-abc_murders.txt to {drive_dir}.")

examples = []
with data_path.open(encoding="utf-8-sig") as data_file:
    for line_number, line in enumerate(data_file, start=1):
        if not line.strip():
            continue
        parts = [part.strip() for part in line.split(" ||| ")]
        if len(parts) != 2 or not all(parts):
            raise ValueError(f"Expected one English-Dutch pair on line {line_number}.")
        source, target = parts
        examples.append(
            {
                "prompt": [{"role": "user", "content": f"English: {source}\nDutch"}],
                "completion": [{"role": "assistant", "content": target}],
            }
        )

dataset = Dataset.from_list(examples)
print(f"Loaded {len(dataset):,} sentence pairs.")

Mounted at /content/drive
Loaded 5,588 sentence pairs.


## 2. Load Llama and add an adapter

We load a 4-bit copy of Llama to fit within the GPU memory available in Colab. LoRA adds weights to selected parts of the model. The count printed below shows how many weights will change during training.

In [3]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU in Colab and run the notebook again.")

set_seed(3407)
base_id = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
base_revision = "bb1d317a108579fb40e646af8924a5e7ec5604b1"
config = AutoConfig.from_pretrained(base_id, revision=base_revision)
config.quantization_config["bnb_4bit_compute_dtype"] = "float16"
config.use_cache = False  # Training recomputes some steps to save memory.

tokenizer = AutoTokenizer.from_pretrained(base_id, revision=base_revision)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_id,
    revision=base_revision,
    config=config,
    dtype=torch.float16,
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model = get_peft_model(
    model,
    LoraConfig(
        task_type="CAUSAL_LM",
        r=16,
        lora_alpha=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    ),
    revision=base_revision,
)
model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 3. Check what the model will learn from

TRL formats each example using Llama’s chat template. It adds the start-of-text token once and uses only the Dutch answer and its end marker to calculate training error. The two lines below show the full tokenized example and the part used for training.

In [4]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs",
        max_steps=60,
        max_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=2e-4,
        warmup_steps=5,
        weight_decay=0.01,
        fp16=True,
        bf16=False,
        optim="adamw_8bit",
        completion_only_loss=True,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=3407,
    ),
)

for row in trainer.train_dataset:
    if row["input_ids"].count(tokenizer.bos_token_id) != 1:
        raise ValueError("Expected one start-of-text token per example.")
    if (
        not any(row["completion_mask"])
        or row["input_ids"][-1] != tokenizer.eos_token_id
    ):
        raise ValueError(
            "A training example is missing its full Dutch answer and end marker."
        )

example = trainer.train_dataset[5]
labels = trainer.data_collator([example])["labels"][0].tolist()
print("Full example:", tokenizer.decode(example["input_ids"]))
print("Scored tokens:", tokenizer.decode([token for token in labels if token != -100]))

Tokenizing train dataset:   0%|          | 0/5588 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5588 [00:00<?, ? examples/s]

Full example: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

English: I may add that they have been " vetted " by my friend Hercule Poirot himself.
Dutch<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Ik kan hier nog bij vermelden, dat een en ander door mijn vriend Hercule Poirot gecontroleerd is.<|eot_id|>
Scored tokens: Ik kan hier nog bij vermelden, dat een en ander door mijn vriend Hercule Poirot gecontroleerd is.<|eot_id|>


## 4. Train and save the adapter

We run 60 training steps to keep the exercise short. When training finishes, the notebook saves the adapter and tokenizer in **My Drive → lora_model**. The evaluation notebook reads this folder directly. The largest file should be `adapter_model.safetensors`, roughly 100 MB for this setup. The base model is downloaded separately when needed.

In [5]:
trainer.train()

model.save_pretrained(adapter_dir, save_embedding_layers=False)
tokenizer.save_pretrained(adapter_dir)
adapter_size = (adapter_dir / "adapter_model.safetensors").stat().st_size
print(f"Saved adapter weights ({adapter_size / 1_000_000:.1f} MB) to {adapter_dir}.")

Step,Training Loss
5,2.078700
10,1.532400
15,1.576700
20,1.393800
25,1.351500
30,1.444500
35,1.363300
40,1.597200
45,1.427400
50,1.345400


Saved adapter weights (97.3 MB) to /content/drive/MyDrive/lora_model.


## 5. Try a translation

The prompt follows the same English/Dutch format as a training example. Run the evaluation notebook next to compare the model with and without the adapter on passages from a different book.

In [6]:
model.eval()
sentence = "Old Lanscombe moved totteringly from room to room, pulling up the blinds."
messages = [{"role": "user", "content": f"English: {sentence}\nDutch"}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=False,
        temperature=None,
        top_p=None,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
print(
    tokenizer.decode(
        output[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
)

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Oud Lanscombe bewoog heftig van kamer naar kamer, terwijl hij de gordijnen opzette.
